# 08_3_8 DQA Scene Phase2 R-SCoLQ Smooth Policy

08_3_7では R-SCoLQ multiplier 自体は効いたが、`1.0 -> 0.15 -> 1.0 -> 0.15` の振動になった。原因は、次roundの罰則を **multiplier後のquality** から計算していたこと。

08_3_8では以下に直す。

```text
raw_score = static R-SCoLQ bbox quality
final_score = raw_score * EMA-smoothed global multiplier * class multiplier
stats quality = raw_score
```

さらに、round policyのanchorは08_3_6由来のSCoLQ scaleではなく、最初のtarget roundのraw R-SCoLQ statsに合わせる。これでscore scaleのズレとオンオフ振動を避ける。

Discord通知セルは最後に置いてある。


In [1]:
from __future__ import annotations

import json
import os
import signal
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "dynamic_quality_aware_classwise_aggregation").exists() and (path / "navigating_data_heterogeneity").exists():
            return path
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
DQA_ROOT = REPO_ROOT / "dynamic_quality_aware_classwise_aggregation"
RUN_SCRIPT = DQA_ROOT / "run_dqa_cwa_fedsto_scene_v2_phase2_rscolq_smooth_policy.py"
EVAL_SCRIPT = DQA_ROOT / "evaluate_scene_protocol.py"
SOURCE_WORK_ROOT = DQA_ROOT / "efficientteacher_dqa08_scene_tri_stage_policy_8h"
RSCOLQ_ROOT = DQA_ROOT / "source_calibrated_localization_quality"
RSCOLQ_MODEL = RSCOLQ_ROOT / "artifacts" / "rscolq_best.joblib"
RSCOLQ_RANKING = RSCOLQ_ROOT / "reports" / "rscolq_model_ranking.csv"
RSCOLQ_DIAG = RSCOLQ_ROOT / "reports" / "rscolq_0836_round_diagnostics.csv"
BASE_WORK_ROOT = DQA_ROOT / "efficientteacher_dqa08_3_8_phase2_rscolq_smooth_policy"
BASE_STATS_ROOT = DQA_ROOT / "stats_dqa08_3_8_phase2_rscolq_smooth_policy"
BASE_LOG_ROOT = DQA_ROOT / "logs_dqa08_3_8_phase2_rscolq_smooth_policy"

for path in [RUN_SCRIPT, EVAL_SCRIPT, SOURCE_WORK_ROOT, RSCOLQ_MODEL]:
    print(path, "exists=", path.exists())

BASE_WORK_ROOT.mkdir(parents=True, exist_ok=True)
BASE_STATS_ROOT.mkdir(parents=True, exist_ok=True)
BASE_LOG_ROOT.mkdir(parents=True, exist_ok=True)

if RSCOLQ_RANKING.exists():
    ranking = pd.read_csv(RSCOLQ_RANKING)
    display(ranking[["candidate", "n_features", "ap50_quality", "ap75_quality", "p50_at_10pct", "proxy_penalty", "selection_score"]])
if RSCOLQ_DIAG.exists():
    diag = pd.read_csv(RSCOLQ_DIAG)
    display(diag[["round", "pseudo_total", "mean_scolq", "global_multiplier", "mean_class_multiplier", "rscolq_proxy", "map50", "map50_95"]])

print("workspace:", BASE_WORK_ROOT)
print("stats:", BASE_STATS_ROOT)
print("logs:", BASE_LOG_ROOT)

/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/run_dqa_cwa_fedsto_scene_v2_phase2_rscolq_smooth_policy.py exists= True
/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/evaluate_scene_protocol.py exists= True
/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_scene_tri_stage_policy_8h exists= True
/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/source_calibrated_localization_quality/artifacts/rscolq_best.joblib exists= True


,candidate,n_features,ap50_quality,ap75_quality,p50_at_10pct,proxy_penalty,selection_score
0,low_conf:hgb,19,0.767833,0.795002,0.653509,0.045,2.480720
1,rank_conf:hgb,21,0.768683,0.794828,0.654162,0.145,2.382401
2,all_proxy:hgb,24,0.768551,0.795472,0.653742,0.295,2.231998
3,safe_no_conf:hgb,18,0.614495,0.512432,0.547675,0.030,1.930348


,round,pseudo_total,mean_scolq,global_multiplier,mean_class_multiplier,rscolq_proxy,map50,map50_95
0,1,770515.0,0.443193,1.000000,0.997977,0.442297,0.382,0.211
1,2,773621.0,0.448643,1.000000,0.996683,0.447155,0.381,0.212
2,3,757997.0,0.450761,1.000000,0.999152,0.450378,0.377,0.211
3,4,760731.0,0.453214,0.977300,0.999490,0.442700,NaN,NaN
4,5,770963.0,0.456014,0.950322,0.998685,0.432790,NaN,NaN
5,6,785763.0,0.460806,0.885269,0.995085,0.405932,NaN,NaN
6,7,795512.0,0.466490,0.805568,0.990184,0.372101,NaN,NaN
7,8,807088.0,0.470166,0.742671,0.987176,0.344701,NaN,NaN
8,9,817731.0,0.476864,0.666687,0.980792,0.311813,NaN,NaN
9,10,819653.0,0.487045,0.597721,0.956666,0.278502,0.357,0.196


workspace: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_8_phase2_rscolq_smooth_policy
stats: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/stats_dqa08_3_8_phase2_rscolq_smooth_policy
logs: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/logs_dqa08_3_8_phase2_rscolq_smooth_policy


## Variant Design

まずは `a_rscolq_smooth_raw_ema_r003` を本命として回す。

- A: raw stats + EMA multiplier + R-SCoLQ high=0.65。08_3_7より少しだけpseudoを絞り、振動を避ける本命。
- B: high=0.70 のprecision寄り。Aでもpseudoが多すぎる場合の比較。
- C: bbox lossを後半ほぼ消す対照。score制御だけではbbox regression driftを止められない場合を見る。

最初はAだけ。見るべき点は `global_multiplier` が急落せず滑らかに動くか、そしてr010が08_3_7/08_3_6より落ちにくいか。


In [2]:
PHASE2_ROUNDS_PER_VARIANT = 10
BATCH_SIZE = 160
WORKERS = 10
GPUS = 2
MASTER_PORT_BASE = 30080
STREAM_TRAIN_OUTPUT = False
MIN_FREE_GIB = 30

SELECTED_VARIANTS: list[str] = ["a_rscolq_smooth_raw_ema_r003"]

COMMON_SMOOTH_ENV = {
    "DQA0838_RSCOLQ_ANCHOR_ROUND": 1,
    "DQA0838_RSCOLQ_GLOBAL_BUDGET_MARGIN": 0.03,
    "DQA0838_RSCOLQ_QUALITY_MARGIN": 0.03,
    "DQA0838_RSCOLQ_GLOBAL_POWER": 1.20,
    "DQA0838_RSCOLQ_QUALITY_POWER": 1.50,
    "DQA0838_RSCOLQ_CLASS_BUDGET_MARGIN": 0.05,
    "DQA0838_RSCOLQ_CLASS_POWER": 0.18,
    "DQA0838_RSCOLQ_MIN_MULTIPLIER": 0.55,
    "DQA0838_RSCOLQ_MULTIPLIER_EMA": 0.70,
}

VARIANTS = [
    {
        "name": "a_rscolq_smooth_raw_ema_r003",
        "description": "08_3_8本命。raw R-SCoLQ statsでpolicyを計算し、global multiplierをEMAで滑らかにする。high=0.65で少しpseudoを絞る。",
        "source_phase1_round": 3,
        "dqa_start_phase": 2,
        "client_train_scope": "all",
        "classwise_blend": 0.060,
        "server_anchor": 14.0,
        "temperature": 2.8,
        "stability_lambda": 0.68,
        "residual_start": 0.12,
        "residual_end": 0.05,
        "min_server_alpha_start": 0.72,
        "min_server_alpha_end": 0.78,
        "env": {
            **COMMON_SMOOTH_ENV,
            "DQA0838_RSCOLQ_LOW": 0.12,
            "DQA0838_RSCOLQ_MID": 0.35,
            "DQA0838_RSCOLQ_HIGH": 0.65,
            "DQA0838_NMS_CONF_THRES": 0.01,
            "DQA0838_TEACHER_LOSS_WEIGHT": 0.30,
            "DQA0838_BOX_LOSS_WEIGHT_START": 0.008,
            "DQA0838_BOX_LOSS_WEIGHT_END": 0.004,
            "DQA0838_OBJ_LOSS_WEIGHT": 0.30,
            "DQA0838_CLS_LOSS_WEIGHT": 0.06,
            "DQA0838_LOW_MID_OBJ_WEIGHT": 0.20,
            "DQA0838_MID_HIGH_OBJ_WEIGHT": 0.62,
            "DQA0838_CLIENT_LR0_START": 0.00085,
            "DQA0838_CLIENT_LR0_END": 0.00018,
            "DQA0838_SERVER_LR0_START": 0.0026,
            "DQA0838_SERVER_LR0_END": 0.0008,
            "DQA0838_MAX_PSEUDO_PER_IMAGE": 70,
            "DQA0838_MAX_PSEUDO_PER_CLASS_IMAGE": 20,
        },
    },
    {
        "name": "b_rscolq_smooth_strict_r003",
        "description": "08_3_8 strict。high=0.70でbbox positiveをさらに絞る。",
        "source_phase1_round": 3,
        "dqa_start_phase": 2,
        "client_train_scope": "all",
        "classwise_blend": 0.050,
        "server_anchor": 16.0,
        "temperature": 3.0,
        "stability_lambda": 0.72,
        "residual_start": 0.09,
        "residual_end": 0.03,
        "min_server_alpha_start": 0.76,
        "min_server_alpha_end": 0.84,
        "env": {
            **COMMON_SMOOTH_ENV,
            "DQA0838_RSCOLQ_LOW": 0.15,
            "DQA0838_RSCOLQ_MID": 0.40,
            "DQA0838_RSCOLQ_HIGH": 0.70,
            "DQA0838_NMS_CONF_THRES": 0.012,
            "DQA0838_TEACHER_LOSS_WEIGHT": 0.26,
            "DQA0838_BOX_LOSS_WEIGHT_START": 0.006,
            "DQA0838_BOX_LOSS_WEIGHT_END": 0.003,
            "DQA0838_OBJ_LOSS_WEIGHT": 0.28,
            "DQA0838_CLS_LOSS_WEIGHT": 0.05,
            "DQA0838_LOW_MID_OBJ_WEIGHT": 0.16,
            "DQA0838_MID_HIGH_OBJ_WEIGHT": 0.52,
            "DQA0838_CLIENT_LR0_START": 0.00075,
            "DQA0838_CLIENT_LR0_END": 0.00014,
            "DQA0838_SERVER_LR0_START": 0.0022,
            "DQA0838_SERVER_LR0_END": 0.0007,
            "DQA0838_MAX_PSEUDO_PER_IMAGE": 60,
            "DQA0838_MAX_PSEUDO_PER_CLASS_IMAGE": 16,
        },
    },
    {
        "name": "c_rscolq_smooth_bbox_decay_r003",
        "description": "08_3_8 bbox decay。raw/EMA policyに加え、bbox lossを後半かなり落とす。",
        "source_phase1_round": 3,
        "dqa_start_phase": 2,
        "client_train_scope": "neck_head",
        "classwise_blend": 0.045,
        "server_anchor": 18.0,
        "temperature": 3.1,
        "stability_lambda": 0.74,
        "residual_start": 0.08,
        "residual_end": 0.02,
        "min_server_alpha_start": 0.78,
        "min_server_alpha_end": 0.86,
        "env": {
            **COMMON_SMOOTH_ENV,
            "DQA0838_RSCOLQ_LOW": 0.12,
            "DQA0838_RSCOLQ_MID": 0.35,
            "DQA0838_RSCOLQ_HIGH": 0.65,
            "DQA0838_NMS_CONF_THRES": 0.01,
            "DQA0838_TEACHER_LOSS_WEIGHT": 0.24,
            "DQA0838_BOX_LOSS_WEIGHT_START": 0.006,
            "DQA0838_BOX_LOSS_WEIGHT_END": 0.001,
            "DQA0838_OBJ_LOSS_WEIGHT": 0.26,
            "DQA0838_CLS_LOSS_WEIGHT": 0.04,
            "DQA0838_LOW_MID_OBJ_WEIGHT": 0.14,
            "DQA0838_MID_HIGH_OBJ_WEIGHT": 0.48,
            "DQA0838_CLIENT_LR0_START": 0.00065,
            "DQA0838_CLIENT_LR0_END": 0.00010,
            "DQA0838_SERVER_LR0_START": 0.0020,
            "DQA0838_SERVER_LR0_END": 0.0006,
            "DQA0838_MAX_PSEUDO_PER_IMAGE": 65,
            "DQA0838_MAX_PSEUDO_PER_CLASS_IMAGE": 18,
        },
    },
]

selected = [v for v in VARIANTS if (not SELECTED_VARIANTS or v["name"] in SELECTED_VARIANTS)]
print("selected:", [v["name"] for v in selected])


selected: ['a_rscolq_smooth_raw_ema_r003']


In [3]:
def variant_work_root(variant: dict) -> Path:
    return BASE_WORK_ROOT / variant["name"]


def variant_stats_root(variant: dict) -> Path:
    return BASE_STATS_ROOT / variant["name"]


def variant_runner_log(variant: dict) -> Path:
    return BASE_LOG_ROOT / f"{variant['name']}_runner.out"


def variant_train_log(variant: dict) -> Path:
    return BASE_LOG_ROOT / f"{variant['name']}_train.log"


def variant_pid_path(variant: dict) -> Path:
    return BASE_LOG_ROOT / f"{variant['name']}.pid"


def variant_env(variant: dict) -> dict[str, str]:
    env = os.environ.copy()
    stats_root = variant_stats_root(variant)
    stats_root.mkdir(parents=True, exist_ok=True)
    env["DQA0838_VARIANT"] = variant["name"]
    env["DQA0838_SOURCE_WORK_ROOT"] = str(SOURCE_WORK_ROOT)
    env["DQA0838_SOURCE_PHASE1_ROUND"] = str(variant["source_phase1_round"])
    env["DQA0838_RSCOLQ_MODEL"] = str(RSCOLQ_MODEL)
    env["DQA0838_CLIENT_TRAIN_SCOPE"] = variant["client_train_scope"]
    env["DQA0838_SERVER_TRAIN_SCOPE"] = "all"
    env["DQA0838_RESIDUAL_START"] = str(variant["residual_start"])
    env["DQA0838_RESIDUAL_END"] = str(variant["residual_end"])
    env["DQA0838_MIN_SERVER_ALPHA_START"] = str(variant["min_server_alpha_start"])
    env["DQA0838_MIN_SERVER_ALPHA_END"] = str(variant["min_server_alpha_end"])
    env["DQA0838_PHASE2_ROUNDS"] = str(PHASE2_ROUNDS_PER_VARIANT)
    env["DQA08_STATS_ROOT"] = str(stats_root.resolve())
    env["DQA08_THRESHOLD_LOG"] = str((stats_root / "phase2_rscolq_smooth_policy_schedule.jsonl").resolve())
    env["DQA_STATS_QUALITY_MODE"] = "rscolq_raw"
    env["DQA0834_STATS_QUALITY_MODE"] = "rscolq_raw"
    env.setdefault("ET_SKIP_AFTER_TRAIN_BEST_VAL", "1")
    for key, value in variant.get("env", {}).items():
        env[key] = str(value)
    return env


def train_cmd(variant: dict, *, stream: bool = STREAM_TRAIN_OUTPUT) -> list[str]:
    cmd = [
        sys.executable,
        str(RUN_SCRIPT),
        "--workspace-root", str(variant_work_root(variant)),
        "--stats-root", str(variant_stats_root(variant)),
        "--phase1-rounds", "0",
        "--phase2-rounds", str(PHASE2_ROUNDS_PER_VARIANT),
        "--batch-size", str(BATCH_SIZE),
        "--workers", str(WORKERS),
        "--gpus", str(GPUS),
        "--master-port", str(MASTER_PORT_BASE + selected.index(variant)),
        "--log-file", str(variant_train_log(variant)),
        "--classwise-blend", str(variant["classwise_blend"]),
        "--server-anchor", str(variant["server_anchor"]),
        "--temperature", str(variant["temperature"]),
        "--stability-lambda", str(variant["stability_lambda"]),
        "--dqa-start-phase", str(variant["dqa_start_phase"]),
    ]
    if stream:
        cmd.append("--stream-train-output")
    return cmd


def history_rows(variant: dict) -> list[dict]:
    path = variant_work_root(variant) / "history.json"
    if not path.exists():
        return []
    return json.loads(path.read_text())


def read_pid(path: Path) -> int | None:
    if not path.exists():
        return None
    try:
        return int(path.read_text().strip())
    except Exception:
        return None


def pid_state(pid: int | None) -> str:
    if not pid:
        return "missing"
    try:
        os.kill(pid, 0)
    except ProcessLookupError:
        return "stopped"
    except PermissionError:
        return "unknown"
    return "running"


def tail_lines(path: Path, n: int = 40) -> list[str]:
    if not path.exists():
        return []
    return path.read_text(errors="replace").splitlines()[-n:]

In [ ]:
# 実行セル。途中で止まっても history/global checkpoint があれば再利用します。
if not selected:
    raise RuntimeError("No selected variants")
if not RSCOLQ_MODEL.exists():
    raise FileNotFoundError(f"Missing R-SCoLQ model: {RSCOLQ_MODEL}")

for variant in selected:
    history = history_rows(variant)
    completed_phase2 = sum(1 for row in history if int(row.get("phase", 0)) == 2)
    print("\n===", variant["name"], "===")
    print(variant["description"])
    print(f"completed_phase2: {completed_phase2}/{PHASE2_ROUNDS_PER_VARIANT}")
    if completed_phase2 >= PHASE2_ROUNDS_PER_VARIANT:
        print("already completed")
        continue

    pid_path = variant_pid_path(variant)
    existing_pid = read_pid(pid_path)
    if pid_state(existing_pid) == "running":
        print("already running pid:", existing_pid)
        continue

    runner_log = variant_runner_log(variant)
    runner_log.parent.mkdir(parents=True, exist_ok=True)
    cmd = train_cmd(variant)
    env = variant_env(variant)
    print("cmd:", " ".join(cmd))
    print("runner_log:", runner_log)
    print("train_log:", variant_train_log(variant))

    with runner_log.open("a", encoding="utf-8", buffering=1) as log:
        log.write("\n" + "=" * 100 + "\n")
        log.write(" ".join(cmd) + "\n")
        process = subprocess.Popen(
            cmd,
            cwd=REPO_ROOT,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        pid_path.write_text(str(process.pid), encoding="utf-8")
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            log.write(line)
        rc = process.wait()
        if rc != 0:
            raise RuntimeError(f"{variant['name']} failed with exit code {rc}. See {runner_log}")
        print("variant completed:", variant["name"])


=== a_rscolq_smooth_raw_ema_r003 ===
08_3_8本命。raw R-SCoLQ statsでpolicyを計算し、global multiplierをEMAで滑らかにする。high=0.65で少しpseudoを絞る。
completed_phase2: 0/10
cmd: /root/micromamba/envs/al_yolov8/bin/python /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/run_dqa_cwa_fedsto_scene_v2_phase2_rscolq_smooth_policy.py --workspace-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_8_phase2_rscolq_smooth_policy/a_rscolq_smooth_raw_ema_r003 --stats-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/stats_dqa08_3_8_phase2_rscolq_smooth_policy/a_rscolq_smooth_raw_ema_r003 --phase1-rounds 0 --phase2-rounds 10 --batch-size 160 --workers 10 --gpus 2 --master-port 30080 --log-file /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/logs_dqa08_3_8_phase2_rscolq_smooth_policy/a_rscolq_smooth_raw_ema_r003_train.log --classwise-blend 0.06 --server-anchor 14.0 --temperature 2.8 --stability-lambda 0.68 --dqa-start-phas

In [ ]:
# 進捗確認セル
rows = []
for variant in selected:
    history = history_rows(variant)
    latest = Path(history[-1]["global"]) if history else variant_work_root(variant) / "global_checkpoints" / "round000_warmup.pt"
    rows.append({
        "variant": variant["name"],
        "pid": read_pid(variant_pid_path(variant)),
        "pid_state": pid_state(read_pid(variant_pid_path(variant))),
        "phase2": f"{sum(1 for row in history if int(row.get('phase', 0)) == 2)}/{PHASE2_ROUNDS_PER_VARIANT}",
        "latest": str(latest),
        "latest_exists": latest.exists(),
        "free_gib": round(shutil.disk_usage(variant_work_root(variant)).free / 1024**3, 2) if variant_work_root(variant).exists() else None,
    })

display(pd.DataFrame(rows))

for variant in selected:
    print("\n===", variant["name"], "runner tail ===")
    print("\n".join(tail_lines(variant_runner_log(variant), 24)))
    sched = variant_stats_root(variant) / "phase2_rscolq_smooth_policy_schedule.jsonl"
    if sched.exists():
        print("\n--- R-SCoLQ schedule tail ---")
        print("\n".join(tail_lines(sched, 12)))

In [ ]:
# early/final checkpoint 評価。r001/r002/r003/r010 を見る。
EVAL_WORKSPACE = variant_work_root(selected[0])
REPORT_DIR = EVAL_WORKSPACE / "validation_reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

checkpoints: list[tuple[str, Path]] = []
seed03 = SOURCE_WORK_ROOT / "global_checkpoints" / "phase1_round003_global.pt"
seed12 = SOURCE_WORK_ROOT / "global_checkpoints" / "phase1_round012_global.pt"
old0834_c02 = DQA_ROOT / "efficientteacher_dqa08_3_4_phase2_feature_quality_sweep" / "c_feature_balanced_neck_head_r003" / "global_checkpoints" / "phase2_round002_global.pt"
old0834_d02 = DQA_ROOT / "efficientteacher_dqa08_3_4_phase2_feature_quality_sweep" / "d_feature_conservative_min_gate_r003" / "global_checkpoints" / "phase2_round002_global.pt"
old0836_a01 = DQA_ROOT / "efficientteacher_dqa08_3_6_phase2_scolq_policy" / "a_scolq_soft_bbox_r003" / "global_checkpoints" / "phase2_round001_global.pt"
old0836_a02 = DQA_ROOT / "efficientteacher_dqa08_3_6_phase2_scolq_policy" / "a_scolq_soft_bbox_r003" / "global_checkpoints" / "phase2_round002_global.pt"
old0836_a10 = DQA_ROOT / "efficientteacher_dqa08_3_6_phase2_scolq_policy" / "a_scolq_soft_bbox_r003" / "global_checkpoints" / "phase2_round010_global.pt"
old0837_a01 = DQA_ROOT / "efficientteacher_dqa08_3_7_phase2_rscolq_policy" / "a_rscolq_round_stable_r003" / "global_checkpoints" / "phase2_round001_global.pt"
old0837_a02 = DQA_ROOT / "efficientteacher_dqa08_3_7_phase2_rscolq_policy" / "a_rscolq_round_stable_r003" / "global_checkpoints" / "phase2_round002_global.pt"
old0837_a10 = DQA_ROOT / "efficientteacher_dqa08_3_7_phase2_rscolq_policy" / "a_rscolq_round_stable_r003" / "global_checkpoints" / "phase2_round010_global.pt"
checkpoints.extend([
    ("p1_r003", seed03),
    ("p1_r012", seed12),
    ("old08_3_4_c_r002", old0834_c02),
    ("old08_3_4_d_r002", old0834_d02),
    ("old08_3_6_a_r001", old0836_a01),
    ("old08_3_6_a_r002", old0836_a02),
    ("old08_3_6_a_r010", old0836_a10),
    ("old08_3_7_a_r001", old0837_a01),
    ("old08_3_7_a_r002", old0837_a02),
    ("old08_3_7_a_r010", old0837_a10),
])

for variant in selected:
    root = variant_work_root(variant) / "global_checkpoints"
    for r in [1, 2, 3, PHASE2_ROUNDS_PER_VARIANT]:
        ckpt = root / f"phase2_round{r:03d}_global.pt"
        if ckpt.exists():
            checkpoints.append((f"{variant['name']}_r{r:03d}", ckpt))

for label, path in checkpoints:
    print(label, path, "exists=", path.exists())

cmd = [
    sys.executable,
    str(EVAL_SCRIPT),
    "--workspace", str(EVAL_WORKSPACE),
    "--splits", "total",
    "--batch-size", "16",
    "--no-plots",
    "--verbose",
]
for label, path in checkpoints:
    if path.exists():
        cmd.extend(["--checkpoint", f"{label}={path}"])

print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO_ROOT, check=True)

summary_csv = REPORT_DIR / "paper_protocol_eval_summary.csv"
if summary_csv.exists():
    df = pd.read_csv(summary_csv)
    out = REPORT_DIR / "paper_protocol_eval_summary_0838_early_total.csv"
    df.to_csv(out, index=False)
    display(df.sort_values(["split", "map50_95", "map50"], ascending=[True, False, False])[["checkpoint_label", "split", "precision", "recall", "map50", "map50_95"]])
    print("saved:", out)
else:
    print("No summary yet:", summary_csv)

In [ ]:
# 4 split final evaluation。上の total 評価で良い checkpoint を BEST_LABELS に入れる。
BEST_LABELS = []

ckpt_map = {label: path for label, path in checkpoints if path.exists()}
if not BEST_LABELS:
    selected_prefixes = tuple(f"{variant['name']}_" for variant in selected)
    BEST_LABELS = [
        label for label in ckpt_map
        if label.startswith(selected_prefixes)
        and (label.endswith("_r001") or label.endswith("_r002") or label.endswith(f"r{PHASE2_ROUNDS_PER_VARIANT:03d}"))
    ]

cmd = [
    sys.executable,
    str(EVAL_SCRIPT),
    "--workspace", str(EVAL_WORKSPACE),
    "--splits", "highway,citystreet,residential,total",
    "--batch-size", "16",
    "--no-plots",
    "--verbose",
]
for label in ["p1_r003", "p1_r012", "old08_3_4_c_r002", "old08_3_4_d_r002", "old08_3_6_a_r001", "old08_3_6_a_r002", "old08_3_6_a_r010", "old08_3_7_a_r001", "old08_3_7_a_r002", "old08_3_7_a_r010", *BEST_LABELS]:
    path = ckpt_map.get(label)
    if path and path.exists():
        cmd.extend(["--checkpoint", f"{label}={path}"])

print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO_ROOT, check=True)

summary_csv = REPORT_DIR / "paper_protocol_eval_summary.csv"
if summary_csv.exists():
    df = pd.read_csv(summary_csv)
    out = REPORT_DIR / "paper_protocol_eval_summary_0838_selected_4splits.csv"
    df.to_csv(out, index=False)
    display(df.sort_values(["split", "map50_95", "map50"], ascending=[True, False, False])[["checkpoint_label", "split", "precision", "recall", "map50", "map50_95"]])
    print("saved:", out)

In [ ]:
# pseudoGT / R-SCoLQ quality / multiplier 推移を確認するセル。
rows = []
for variant in selected:
    stats_root = variant_stats_root(variant)
    for r in range(1, PHASE2_ROUNDS_PER_VARIANT + 1):
        path = stats_root / f"phase2_round{r:03d}.json"
        if not path.exists():
            continue
        data = json.loads(path.read_text())
        counts = [0.0] * 10
        qsum = [0.0] * 10
        objsum = [0.0] * 10
        clssum = [0.0] * 10
        for client in data.get("clients", []):
            for i, value in enumerate(client.get("counts", [])[:10]):
                counts[i] += float(value)
            for i, value in enumerate(client.get("quality_sums", [])[:10]):
                qsum[i] += float(value)
            for i, value in enumerate(client.get("objectness_sums", [])[:10]):
                objsum[i] += float(value)
            for i, value in enumerate(client.get("class_confidence_sums", [])[:10]):
                clssum[i] += float(value)
        total = sum(counts)
        rows.append({
            "variant": variant["name"],
            "round": r,
            "pseudo_total": total,
            "mean_rscolq_quality": sum(qsum) / total if total else None,
            "mean_objectness": sum(objsum) / total if total else None,
            "mean_class_conf": sum(clssum) / total if total else None,
            "active_classes": sum(1 for x in counts if x > 0),
            "person_count": counts[0],
            "car_count": counts[2],
            "traffic_light_count": counts[7],
            "traffic_sign_count": counts[8],
        })

stats_df = pd.DataFrame(rows)
if not stats_df.empty:
    display(stats_df)
    display(stats_df.groupby("variant")[["pseudo_total", "mean_rscolq_quality", "mean_objectness", "active_classes"]].agg(["min", "max", "mean"]))
else:
    print("No pseudo stats yet")

for variant in selected:
    state_path = variant_work_root(variant) / "dqa_cwa_state.json"
    if state_path.exists():
        state = json.loads(state_path.read_text())
        gate = state.get("phase2_rscolq_smooth_policy", [])
        if gate:
            print("\n", variant["name"])
            display(pd.DataFrame(gate))
    sched = variant_stats_root(variant) / "phase2_rscolq_smooth_policy_schedule.jsonl"
    if sched.exists():
        sched_rows = [json.loads(line) for line in sched.read_text().splitlines() if line.strip()]
        if sched_rows:
            display(pd.DataFrame(sched_rows))

## 読み方

08_3_8で一番見るべきなのは、08_3_7で起きた `global_multiplier = 1.0 / 0.15` の振動が消えるか。

- `quality_mode` は `rscolq_raw` なので、statsの `quality_sums` は multiplier後ではなく raw R-SCoLQ を表す。
- `phase2_rscolq_smooth_policy` の `rscolq_target_multiplier` と `rscolq_global_multiplier` を見る。targetが動いてもglobalがEMAで滑らかなら成功。
- r001/r002 が08_3_6/08_3_7と同程度で、r010の `mAP50-95` が0.196より上なら、長期安定化として前進。
- もしr001から悪ければ `RSCOLQ_HIGH=0.65` が厳しすぎる。もしr010だけ落ちるならbbox loss decay型のCへ進む。


In [ ]:
import sys
sys.path.insert(0, "/app/Object_Detection")

from notebook_notify import notify_discord

notify_discord("０８＿３＿８終わった", title="実行終了")
